In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# Dataset: CO2 Emissions by Country 2000-2022
# Source: Our World in Data (https://ourworldindata.org/co2-emissions)

df = pd.read_csv(r"C:\Users\HP\Documents\GitHub\data-viz-class-material\data\co2_emissions.csv")
print(f"Loaded: {len(df)} rows | Countries: {df['Country'].nunique()} | Years: {df['Year'].min()}-{df['Year'].max()}")
print(df.head())

In [ ]:
# Explore before building.

print("Countries:", df['Country'].unique())
print("\nCO2 range:", df['CO2_Mt'].min(), "to", df['CO2_Mt'].max(), "Mt")
print("\nRegional averages (2022):")
print(df[df['Year']==2022].groupby('Region')['CO2_Mt'].mean().sort_values(ascending=False).round(1))

In [ ]:
# Task 1 — Multi-series line with highlight

asia_df = df[df['Region'] == 'Asia']

highlight_country = "China"  # you can change this

fig = go.Figure()

# Loop through countries
for country in asia_df['Country'].unique():
    country_data = asia_df[asia_df['Country'] == country]
    
    if country == highlight_country:
        fig.add_trace(go.Scatter(
            x=country_data['Year'],
            y=country_data['CO2_Mt'],
            mode='lines',
            line=dict(color='#32213a', width=3),
            name=country
        ))
        
        # Add direct label at end
        fig.add_annotation(
            x=country_data['Year'].max(),
            y=country_data['CO2_Mt'].iloc[-1],
            text=country,
            showarrow=False,
            font=dict(size=12, color='#32213a'),
            xanchor='left'
        )
        
    else:
        fig.add_trace(go.Scatter(
            x=country_data['Year'],
            y=country_data['CO2_Mt'],
            mode='lines',
            line=dict(color='#DDDDDD', width=1),
            hoverinfo='skip',
            showlegend=False
        ))

# Layout styling 
fig.update_layout(
    title="China dominates CO₂ emissions growth in Asia since 2000",
    plot_bgcolor='white',
    paper_bgcolor='white',
    font=dict(family='Arial', size=12),
    margin=dict(l=40, r=40, t=60, b=40),
)

# Remove clutter
fig.update_xaxes(showgrid=False, zeroline=False)
fig.update_yaxes(showgrid=False, zeroline=False)

fig.show()



In [ ]:
# Task 2 — Slopegraph: Regional Change 2000 vs 2022
# ------------------------------------------------

# Step 1: Aggregate by Region + Year
region_avg = df.groupby(['Region', 'Year'])['CO2_Mt'].mean().reset_index()

# Step 2: Filter only 2000 and 2022
slope_df = region_avg[region_avg['Year'].isin([2000, 2022])]

# Step 3: Pivot for easier plotting
pivot_df = slope_df.pivot(index='Region', columns='Year', values='CO2_Mt').reset_index()
pivot_df.columns = ['Region', 'CO2_2000', 'CO2_2022']

# Step 4: Define color based on increase/decrease
pivot_df['Change'] = pivot_df['CO2_2022'] - pivot_df['CO2_2000']

def get_color(change):
    return '#2E8B57' if change > 0 else '#0b132b'  # green vs red

# Step 5: Build slopegraph
fig = go.Figure()

for _, row in pivot_df.iterrows():
    color = get_color(row['Change'])
    
    fig.add_trace(go.Scatter(
        x=[2000, 2022],
        y=[row['CO2_2000'], row['CO2_2022']],
        mode='lines+markers',
        line=dict(color=color, width=2),
        marker=dict(size=6),
        showlegend=False
    ))
    
    # Left label (2000)
    fig.add_annotation(
        x=2000,
        y=row['CO2_2000'],
        text=f"{row['Region']} ({row['CO2_2000']:.0f})",
        showarrow=False,
        xanchor='right',
        font=dict(size=11)
    )
    
    # Right label (2022)
    fig.add_annotation(
        x=2022,
        y=row['CO2_2022'],
        text=f"{row['CO2_2022']:.0f}",
        showarrow=False,
        xanchor='left',
        font=dict(size=11)
    )

# Step 6: Clean layout (no clutter)
fig.update_layout(
    title="Asia shows the largest rise in average CO₂ emissions since 2000",
    plot_bgcolor='white',
    paper_bgcolor='white',
    font=dict(family='Arial', size=12),
    margin=dict(l=80, r=80, t=60, b=40),
)

# Remove y-axis labels (important for slopegraph)
fig.update_yaxes(showticklabels=False, showgrid=False, zeroline=False)
fig.update_xaxes(
    tickvals=[2000, 2022],
    showgrid=False,
    zeroline=False
)

fig.show()

